In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer
import numpy as np

In [2]:
# 🔹 Ustawienie losowości dla powtarzalnych wyników
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# 2️⃣ Sieć neuronowa end-to-end
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 4),
            nn.ReLU(),
            nn.Linear(4, 2),
            nn.Sigmoid() 
        )

    def forward(self, x):
        return self.net(x)
    
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 8),
            nn.ReLU(),
            nn.Linear(8, 6),
            nn.ReLU(),
            nn.Linear(6, 2)
        )

    def forward(self, x):
        return self.net(x)
    
# 3️⃣ Autoenkoder
class Autoencoder(nn.Module):
    def __init__(self, encoding_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(30, 20),
            nn.ReLU(inplace=True),
            nn.Linear(20, 10),
            nn.ReLU(inplace=True),
            nn.Linear(10, encoding_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 10),
            nn.ReLU(inplace=True),
            nn.Linear(10, 20),
            nn.ReLU(inplace=True),
            nn.Linear(20, 30)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [78]:
# 1️⃣ Dane – zbiór Iris
iris = load_breast_cancer()
X = iris.data
y = iris.target

[1 1 0 1 1 1 0 1 0 1 1 1 0 1 0 0 1 1 0 1 0 0 0 1 1 1 0 1 1 0 1 0 1 1 1 0 1
 0 1 1 0 0 1 1 0 1 0 0 1 0 0 1 1 0 0 0 1 1 1 1 0 1 0 0 0 0 1 1 1 1 1 1 1 1
 0 0 1 1 0 1 1 1 1 1 0 1 1 0 0 1 0 1 0 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 0 1
 1 0 1 0 0 0 1 0 1 1 0 0 0 1 1 1 1 1 1 1 0 1 1 1 0 1 1 0 0 1 0 1 0 0 1 1 0
 1 0 0 1 0 0 1 1 0 1 0 1 1 0 1 1 0 0 0 1 1 1 0 0 1 0 0 1 1 1 0 1 0 0 0 0 1
 1 0 1 1 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 1 1 1 1 0 1 0 1 1 1 1 1 0 0 0 1
 1 0 1 1 0 0 0 0 1 1 0 0 1 1 1 0 0 0 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 0 1
 1 1 1 1 1 0 1 1 0 1 1 0 0 0 1 0 0 1 0 1 1 1 1 0 1]


In [60]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [92]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [93]:
mlp = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.01)

for epoch in range(500):
    optimizer.zero_grad()
    outputs = mlp(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

In [94]:
with torch.no_grad():
    preds = mlp(X_test_t).argmax(dim=1)
    train_preds = mlp(X_train_t).argmax(dim=1)
mlp_acc = accuracy_score(y_test, preds.numpy())
mlp_acc_train = accuracy_score(y_train, train_preds.numpy())

In [174]:
mlp_2 = MLP2()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_2.parameters(), lr=0.01)

for epoch in range(500):
    optimizer.zero_grad()
    outputs = mlp_2(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:  # co 50 epok
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

with torch.no_grad():
    preds = mlp_2(X_test_t).argmax(dim=1)
    train_preds = mlp_2(X_train_t).argmax(dim=1)

print(y_train_t)
mlp_acc_2 = accuracy_score(y_test, preds.numpy())
mlp_acc_train_2 = accuracy_score(y_train, train_preds.numpy())
print(f"Dokładność MLP end-to-end: {mlp_acc_2:.3f}")
print(f"Dokładność MLP end-to-end train: {mlp_acc_train_2:.3f}")

Epoch 000 | Loss: 0.8200
Epoch 050 | Loss: 0.4555
Epoch 100 | Loss: 0.4026
Epoch 150 | Loss: 0.3924
Epoch 200 | Loss: 0.3839
Epoch 250 | Loss: 0.3834
Epoch 300 | Loss: 0.3787
Epoch 350 | Loss: 0.3680
Epoch 400 | Loss: 0.3881
Epoch 450 | Loss: 0.3649
tensor([1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1,
        1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0,
        1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1,
        1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1,
        1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0,
        0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1,
        1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0,
        0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1,
        1, 1, 0, 0, 0, 1, 1, 0

In [95]:
autoencoder = Autoencoder(encoding_dim=3)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001, weight_decay=1e-5)

for epoch in range(2500):
    optimizer.zero_grad()
    outputs = autoencoder(X_train_t)
    loss = criterion(outputs, X_train_t)
    loss.backward()
    optimizer.step()

In [96]:
# 4️⃣ Ekstrakcja cech z enkodera
with torch.no_grad():
    X_train_encoded = autoencoder.encoder(X_train_t).numpy()
    X_test_encoded = autoencoder.encoder(X_test_t).numpy()

In [97]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

(284, 3)
(285, 3)


In [98]:
# 5️⃣ Klasyfikator SVM
svm = SVC(kernel='rbf', gamma='scale')
svm.fit(X_train_encoded, y_train)
y_pred = svm.predict(X_test_encoded)
svm_acc = accuracy_score(y_test, y_pred)

In [100]:
# 5️⃣ Klasyfikator SVM
svm = SVC(kernel='rbf', gamma='scale')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
svm_acc_raw = accuracy_score(y_test, y_pred)

In [102]:
# 6️⃣ Wyniki
print(f"Dokładność MLP end-to-end: {mlp_acc:.3f}")
print(f"Dokładność MLP end-to-end train: {mlp_acc_train:.3f}")
print(f"Dokładność SVM na cechach z autoenkodera: {svm_acc:.3f}")
print(f"Dokładność SVM: {svm_acc_raw:.3f}")

Dokładność MLP end-to-end: 0.656
Dokładność MLP end-to-end train: 0.599
Dokładność SVM na cechach z autoenkodera: 0.930
Dokładność SVM: 0.926


In [23]:
# wymagane: numpy, sklearn, torch
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Przygotowanie: konwersja do tensora jeśli masz numpy
# X_train_np, X_test_np, y_train, y_test - numpy arrays
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

# 1) Rekonstrukcja i błąd MSE (train/test)
autoencoder.eval()
with torch.no_grad():
    X_train_rec_t = autoencoder(X_train_t)
    X_test_rec_t  = autoencoder(X_test_t)

X_train_rec = X_train_rec_t.cpu().numpy()
X_test_rec  = X_test_rec_t.cpu().numpy()

mse_train = mean_squared_error(X_train, X_train_rec)
mse_test  = mean_squared_error(X_test, X_test_rec)
print(f"MSE train: {mse_train:.6f}, MSE test: {mse_test:.6f}")

# 2) Per-feature MSE i R^2 (ułatwia zlokalizowanie problemów)
per_feature_mse = ((X_test - X_test_rec)**2).mean(axis=0)
per_feature_r2  = [r2_score(X_test[:,i], X_test_rec[:,i]) for i in range(X_test.shape[1])]
print("Per-feature MSE:", np.round(per_feature_mse,6))
print("Per-feature R2:", np.round(per_feature_r2,4))

# 3) Ekstrakcja cech (latent) i konwersja do numpy
with torch.no_grad():
    X_train_latent = autoencoder.encoder(X_train_t).cpu().numpy()
    X_test_latent  = autoencoder.encoder(X_test_t).cpu().numpy()

print("Latent shape:", X_train_latent.shape)

# 4) Downstream classification: prosty LogisticRegression + SVM
# logistic
log = LogisticRegression(max_iter=100)
log.fit(X_train_latent, y_train)
y_pred_log = log.predict(X_test_latent)
acc_log = accuracy_score(y_test, y_pred_log)

# svm
svm = SVC(kernel='rbf', gamma='scale')
svm.fit(X_train_latent, y_train)
y_pred_svm = svm.predict(X_test_latent)
acc_svm = accuracy_score(y_test, y_pred_svm)

print(f"Downstream accuracy (Logistic on latent): {acc_log:.3f}")
print(f"Downstream accuracy (SVM on latent): {acc_svm:.3f}")

# 5) Porównanie z klasyfikatorem na surowych danych (baseline)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_raw = sc.fit_transform(X_train)
X_test_raw  = sc.transform(X_test)

log_raw = LogisticRegression(max_iter=100).fit(X_train_raw, y_train)
acc_log_raw = accuracy_score(y_test, log_raw.predict(X_test_raw))
print(f"Logistic on raw features: {acc_log_raw:.3f}")

# Wnioski pomocnicze (prosty)
if mse_test > 5*mse_train:
    print("Uwaga: możliwy overfitting (mse_test >> mse_train).")
if acc_svm < acc_log_raw:
    print("Enkoder daje gorsze cechy niż surowe dane dla SVM — rozważ zmiany w trainingu/architekturze.")


MSE train: 0.216533, MSE test: 0.337637
Per-feature MSE: [0.117517 0.732916 0.110688 0.082195 0.532122 0.139499 0.107024 0.144544
 0.68861  0.281557 0.187498 0.766744 0.174019 0.088802 0.369535 0.303466
 0.221795 0.482563 0.706833 0.298435 0.097189 0.837567 0.091113 0.099732
 0.589495 0.325376 0.249186 0.154165 0.734346 0.414587]
Per-feature R2: [0.8795 0.3051 0.8886 0.9119 0.4899 0.8671 0.8958 0.874  0.2976 0.661
 0.7739 0.3218 0.7836 0.8711 0.5653 0.5556 0.505  0.4221 0.3528 0.4701
 0.9063 0.2357 0.9137 0.9027 0.3355 0.7087 0.743  0.8532 0.119  0.6214]
Latent shape: (455, 2)
Downstream accuracy (Logistic on latent): 0.965
Downstream accuracy (SVM on latent): 0.965
Logistic on raw features: 0.974
Enkoder daje gorsze cechy niż surowe dane dla SVM — rozważ zmiany w trainingu/architekturze.
